# 💎 Diamond Price Prediction — Multiple Linear Regression
**Dataset:** Kaggle Diamonds (ggplot2 / Shivam2503)  
**Goal:** Predict diamond price using physical + quality features via MLR

---

## 1. Imports & Setup

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

from data_generator import generate_diamonds
from utils import encode_features, FEATURE_COLS, TARGET_COL, FEATURE_LABELS

plt.rcParams.update({'figure.dpi': 120, 'axes.facecolor': '#16171e',
                     'figure.facecolor': '#0b0c10', 'text.color': '#c9d6ff',
                     'axes.labelcolor': '#8892a4', 'xtick.color': '#c9d6ff',
                     'ytick.color': '#c9d6ff', 'axes.edgecolor': '#2d2f42'})
print('✅ Setup complete')

## 2. Load Data

In [ ]:
df = generate_diamonds(n=5000)
df = encode_features(df)
print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.describe().round(2)

In [ ]:
print('Missing values:')
print(df.isnull().sum())

## 3. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Feature Distributions', color='#c9d6ff', fontsize=14)

num_cols = ['carat', 'depth', 'table', 'x', 'y', 'z']
for ax, col in zip(axes.flat, num_cols):
    ax.hist(df[col], bins=40, color='#a18cd1', edgecolor='none', alpha=0.85)
    ax.set_title(col, color='#c9d6ff')

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sc = ax.scatter(df['carat'], df['price'], c=df['clarity_num'],
                cmap='cool', alpha=0.4, s=8)
plt.colorbar(sc, ax=ax, label='Clarity Grade')
ax.set_xlabel('Carat'); ax.set_ylabel('Price ($)')
ax.set_title('Carat vs Price (coloured by Clarity)')
plt.show()

In [ ]:
num_cols_corr = ['carat','depth','table','x','y','z','price','cut_num','color_num','clarity_num']
corr = df[num_cols_corr].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, ax=ax, annot=True, fmt='.2f', cmap='RdPu',
            linewidths=0.4, linecolor='#0b0c10', annot_kws={'size':8})
ax.set_title('Correlation Matrix', color='#c9d6ff')
plt.tight_layout()
plt.show()

## 4. Feature Engineering

In [ ]:
print('Feature columns used for training:')
for f in FEATURE_COLS:
    print(f'  {f:15s} → {FEATURE_LABELS[f]}')

## 5. Train / Test Split

In [ ]:
X = df[FEATURE_COLS]
y = df[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
print(f'Train: {len(X_train):,}  |  Test: {len(X_test):,}')

## 6. Multiple Linear Regression Model

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print('✅ Model trained')
print(f'  Intercept : {model.intercept_:.2f}')
print()
for feat, coef in zip(FEATURE_COLS, model.coef_):
    print(f'  {feat:15s}  β = {coef:+.4f}')

## 7. Evaluation Metrics

In [ ]:
r2   = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae  = mean_absolute_error(y_test, y_pred)
cv   = cross_val_score(model, X, y, cv=5, scoring='r2')

print(f'R²   (test)  : {r2:.4f}')
print(f'RMSE (test)  : ${rmse:,.2f}')
print(f'MAE  (test)  : ${mae:,.2f}')
print(f'R²   (5-CV)  : {cv.mean():.4f} ± {cv.std():.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Actual vs Predicted
axes[0].scatter(y_test, y_pred, alpha=0.3, s=7, color='#a18cd1')
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
axes[0].plot(lims, lims, 'r--', linewidth=1.5, label='Perfect fit')
axes[0].set_xlabel('Actual'); axes[0].set_ylabel('Predicted')
axes[0].set_title(f'Actual vs Predicted  (R²={r2:.3f})')
axes[0].legend()

# Residuals
residuals = np.array(y_test) - y_pred
axes[1].hist(residuals, bins=50, color='#6ee7b7', edgecolor='none', alpha=0.85)
axes[1].axvline(0, color='#fca5a5', linewidth=1.5, linestyle='--')
axes[1].set_xlabel('Residual ($)'); axes[1].set_ylabel('Frequency')
axes[1].set_title('Residual Distribution')

plt.tight_layout()
plt.show()

## 8. Save Model

In [ ]:
import pickle, os

os.makedirs('../models', exist_ok=True)
with open('../models/mlr_model.pkl', 'wb') as f:
    pickle.dump(model, f)

meta = {'r2': round(r2,4), 'rmse': round(rmse,2), 'mae': round(mae,2),
        'cv_r2': round(cv.mean(),4), 'cv_r2_std': round(cv.std(),4),
        'intercept': round(float(model.intercept_),4)}

with open('../models/model_meta.pkl', 'wb') as f:
    pickle.dump(meta, f)

print('✅ Model & metadata saved to ../models/')